In [13]:
import re
import os

filePath = os.path.normpath(os.path.join("..", "design", "2layer", "rightside")) + os.sep
fileName = "rightside.kicad_pcb"
newFileName = "rightside_new.kicad_pcb"
print(os.getcwd())
fullpath = os.path.join(filePath, fileName)
if os.path.exists(fullpath):
    print("File exists")
else:
    print(f"File {fullpath} does not exist")
    allFiles = os.listdir(os.path.normpath(os.path.join("..", "design", "2layer", "rightside")))
    print("All files in directory:")
    for f in allFiles:
        print(f)
# ...existing code...

/mnt/shared/Arch/Desktop/Personal/Projects/PCB/Keyboard/src
File exists


In [14]:
pattern = re.compile(r'uuid.+\n\s+\(at ([0-9\.]+) ([0-9\.]+)')
file = open(fullpath, 'r')
fileText = file.read()
file.close()
matches = pattern.findall(fileText)
print(f"There are {len(matches)} matches")

There are 238 matches


In [15]:
# Mirror footprints
# Find largest x
maxX = 0
for match in matches:
    x = float(match[0])
    if x > maxX:
        maxX = x
print(f"Largest x is {maxX}")
# subtract largest x from all x and invert
newFileText = fileText
for match in matches:
    x = float(match[0])
    y = float(match[1])
    newX = maxX - x
    newFileText = newFileText.replace(f"(at {x} {y}", f"(at {newX} {y}")


Largest x is 166.462685


In [16]:
# Mirror edge cuts
edgecutLine = re.compile(r'gr_line\n\s+\(start ([0-9\.]+) ([0-9\.]+)\)\n\s+\(end ([0-9\.]+) ([0-9\.]+)')
matches = edgecutLine.findall(newFileText)
print(f"There are {len(matches)} edge cut lines")
for match in matches:
    x1 = float(match[0])
    y1 = float(match[1])
    x2 = float(match[2])
    y2 = float(match[3])
    newX1 = maxX - x1
    newX2 = maxX - x2
    newFileText = newFileText.replace(f"(start {x1} {y1}", f"(start {newX1} {y1}")
    newFileText = newFileText.replace(f"(end {x2} {y2}", f"(end {newX2} {y2}")

There are 6 edge cut lines


In [17]:
# Mirror circles
circlecutlines = re.compile(r'gr_circle\n\s+\(center ([0-9\.]+) ([0-9\.]+)\)\n\s+\(end ([0-9\.]+) ([0-9\.]+)')
matches = circlecutlines.findall(newFileText)
print(f"There are {len(matches)} circle cut lines")
for match in matches:
    x1 = float(match[0])
    y1 = float(match[1])
    x2 = float(match[2])
    y2 = float(match[3])
    newX1 = maxX - x1
    newX2 = maxX - x2
    newFileText = newFileText.replace(f"(center {x1} {y1}", f"(center {newX1} {y1}")
    newFileText = newFileText.replace(f"(end {x2} {y2}", f"(end {newX2} {y2}")


There are 1 circle cut lines


In [18]:
# Mirror vias
via = re.compile(r'via\n\s+\(at ([0-9\.]+) ([0-9\.]+)')
matches = via.findall(newFileText)
print(f"There are {len(matches)} vias")
for match in matches:
    x = float(match[0])
    y = float(match[1])
    newX = maxX - x
    newFileText = newFileText.replace(f"(at {x} {y}", f"(at {newX} {y}")

There are 0 vias


In [19]:
# mirror zones
zone = re.compile(r'\(polygon\n\s+\(pts\n\s+((?:\(xy [0-9\.]+ [0-9\.]+\)\s*)+)')
matches = zone.findall(newFileText)
print(f"There are {len(matches)} zones")
for match in matches:
    points = re.findall(r'\(xy ([0-9\.]+) ([0-9\.]+)\)', match)
    newPoints = []
    for point in points:
        x = float(point[0])
        y = float(point[1])
        newX = maxX - x
        newPoints.append(f"(xy {newX} {y})")
    newFileText = newFileText.replace(match, " ".join(newPoints))

There are 4 zones


In [ ]:
#mirror user 1
# example
	# (gr_rect
	# 	(start 146.378373 56.132368)
	# 	(end 164.378373 83.132368)
	# 	(stroke
	# 		(width 0.1)
	# 		(type solid)
	# 	)
	# 	(fill no)
	# 	(layer "User.1")
	# 	(uuid "e2828183-c6d9-4554-b188-bb0d0c50197e")
	# )
user1 = re.compile(r'\(user\s+"1"\n\s+\(at ([0-9\.]+) ([0-9\.]+)')


In [23]:
filepath = os.path.join(filePath, newFileName)
file = open(filepath, 'w')
file.write(newFileText)
file.close()